# k-mers, GC content and GC skew

*bioinformatics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/kmer_gc_skew)

A DNA sequence becomes a one-hot $(L, 4)$ tensor. Counting every word of $k$ letters is one einsum over unfolded windows (a 1-D convolution), GC content and GC skew are window sums, and the minimum of the cumulative skew points to where replication starts on a synthetic circular chromosome with a planted origin and terminus.

**Vector form**

$$N(w) = \#\{n : s_n s_{n+1}\cdots s_{n+k-1} = w\},\qquad \mathrm{GC} = \frac{G + C}{\ell},\qquad \mathrm{skew} = \frac{G - C}{G + C},\qquad S(x) = \sum_{n \le x} \big([s_n = \mathrm{G}] - [s_n = \mathrm{C}]\big)$$

**Index (tensor) form**

$$m_n = U_{naj}\,P_{ja},\quad U_{naj} = X_{n+j,\,a} \qquad N_m = \sum_n [m_n = m] \qquad n_{wa} = \sum_i X_{w\Delta + i,\,a} \qquad S_n = \sum_{n' \le n} X_{n'a}\,c_a$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X_{na}$ | `one_hot(seq)` | (L, 4) | position $n$, letter $a$ ∈ (A, C, G, T): 1 where the base is $a$ |
| $U_{naj}$ | `X.unfold(0, k, 1)` | (L−k+1, 4, k) | every window of $k$ letters, without copying memory |
| $P_{ja}$ | `place_values(k)` | (k, 4) | $a \cdot 4^{k-1-j}$: the word read as a number in base 4 (AAA = 0, …, TTT = $4^k-1$) |
| $m_n = U_{naj} P_{ja}$ | `einsum('naj,ja->n', U, P)` | (L−k+1,) | repeated $j$ and $a$ ⇒ summed: the index of the $k$-mer that starts at $n$ |
| $N_m$ | `bincount(m, minlength=4**k)` | (4^k,) | the $k$-mer spectrum; the counts add up to $L-k+1$ |
| $n_{wa}$ | `einsum('wai->wa', X.unfold(0, ℓ, Δ))` | (W, 4) | repeated $i$ ⇒ summed: base counts in window $w$ of $\ell$ bases, one every $\Delta = \ell/2$ |
| $\mathrm{GC}_w,\ \mathrm{skew}_w$ | `gc_content(n, g), gc_skew(n, g, c)` | (W,) | $(n_G + n_C)/\ell$ and $(n_G - n_C)/(n_G + n_C)$ |
| $S_n$ | `cumsum(einsum('na,a->n', X, c))` | (L,) | $c = (0, -1, +1, 0)$: running count of G minus C; argmin = origin, argmax = terminus |

Most bacterial chromosomes are circular and are copied from a single origin of replication, with two replication forks running in opposite directions until they meet at the terminus. The two halves are copied differently (one strand continuously as the leading strand, the other in pieces), and over evolutionary time the leading strand tends to collect slightly more G than C. Here that bias is planted: from the origin to the terminus G is favoured by the factor $(1+\text{skew})/(1-\text{skew})$, on the other half C is. The skew of one window is noisy, but the running sum $S_n$ falls all the way to the origin and climbs all the way to the terminus, so its minimum and maximum stand out even when the bias is only a few percent. Real genomes show the same V shape, though less cleanly, because genes, repeats and rearrangements add their own composition. The two low-GC stretches on the inner ring imitate genomic islands, DNA picked up from elsewhere that often keeps a different base composition. The $k$-mer spectrum is the starting point of genome assembly and of many alignment-free comparisons; with this random genome its colours simply follow the GC content. Try a smaller skew, a smaller genome or a larger $k$ (more words, fewer counts each).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/kmer_gc_skew/main.py
"""Main loop: parameters → synthetic genome → one-hot tensor → k-mers, GC skew, origin → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    L = int(params["length_kb"] * 1000)
    k, size = params["k"], params["window"]
    seq, origin, terminus = tensors.synthetic_genome(
        L, params["origin"], params["gc"], params["skew"], params["seed"], dev
    )
    X = tensors.one_hot(seq)  # (L, 4)
    is_gc, g_minus_c = tensors.gc_strand_weights(dev)  # (4,), (4,)

    counts = solve.count_kmers(X, tensors.place_values(k, dev))  # (4^k,)
    win = solve.windows(X, size, is_gc, g_minus_c)  # dict of (W,) tensors
    S, found_ori, found_ter = solve.find_origin(X, g_minus_c)  # (L,), int, int

    names = tensors.kmer_names(k)
    top = counts.argsort(descending=True)[:5].tolist()
    print(f"genome L = {L:,} bp (circular)   ·   GC content {X[:, 1:3].sum().item() / L:.3f}")
    print(f"{k}-mers: {counts.sum().item():,} counted = L - k + 1 = {L - k + 1:,}   ·   4^k = {4**k} words")
    print("most common: " + ", ".join(f"{names[m]} {counts[m].item()}" for m in top))
    print(f"windows of {size} bp: GC from {win['gc'].min().item():.3f} to {win['gc'].max().item():.3f}")
    for name, planted, found in [("origin", origin, found_ori), ("terminus", terminus, found_ter)]:
        off = solve.circular_distance(planted, found, L)
        print(f"planted {name:<8} {planted:>9,}   found {found:>9,}   off by {off:,} bp")
    return plot.animate(
        win, S, counts, names, L, (origin, terminus), (found_ori, found_ter), out_dir / "kmer_gc_skew.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/kmer_gc_skew/equation.py
r"""Sequence → tensor translation.

A DNA sequence of length L becomes a one-hot matrix X_{na} (position n, letter a ∈ {A, C, G, T}).

k-mers. Unfolding gives every window of k letters, U_{naj} = X_{n+j, a}. Reading each window as a number in base 4,

    m_n = U_{naj} P_{ja}          (j and a summed),     P_{ja} = a · 4^(k-1-j)

and the k-mer spectrum counts how often each number appears, N_m = Σ_n [m_n = m]. This is a 1-D convolution of X
with a fixed (k, 4) kernel, exactly what a convolutional layer computes.

GC content and skew in windows of w bases, starting every `step` bases (index w for the window, i inside it):

    n_{wa} = U^{(w)}_{wai}        (i summed: base counts per window)
    GC_w = (n_{wG} + n_{wC}) / w,      skew_w = (n_{wG} - n_{wC}) / (n_{wG} + n_{wC})

Cumulative skew. d_n = X_{na} c_a with c = (0, -1, +1, 0) is +1 for G, -1 for C, and S_n = Σ_{n' ≤ n} d_{n'}.
S falls from the terminus to the origin (lagging strand, C-rich) and rises from the origin to the terminus (leading
strand, G-rich), so its minimum marks the replication origin and its maximum the terminus.
"""

import torch


def unfold(X: torch.Tensor, size: int, step: int = 1) -> torch.Tensor:
    """U_{naj} = X_{n·step + j, a}: all windows of `size` letters.   (L, 4) → (n_windows, 4, size), no copy."""
    return X.unfold(0, size, step)


def kmer_index(U: torch.Tensor, P: torch.Tensor) -> torch.Tensor:
    r"""m_n = U_{naj} P_{ja}  (j and a summed): the base-4 number of every k-mer.   (n, 4, k), (k, 4) → (n,) long."""
    return torch.einsum("naj,ja->n", U, P).round().long()


def kmer_counts(m: torch.Tensor, k: int) -> torch.Tensor:
    """N_m = Σ_n [m_n = m]: how often each of the 4^k words occurs.   (n,) → (4^k,)"""
    return torch.bincount(m, minlength=4**k)


def circular(X: torch.Tensor, size: int, step: int) -> torch.Tensor:
    """Append the first size - step bases so windows wrap round the circular chromosome.   (L, 4) → (L+size-step, 4)"""
    return torch.cat([X, X[: size - step]])


def window_base_counts(X: torch.Tensor, size: int, step: int) -> torch.Tensor:
    """n_{wa} = U_{wai}  (i summed): how many A, C, G, T are in each window.   (L, 4) → (n_windows, 4)"""
    return torch.einsum("wai->wa", unfold(X, size, step))


def gc_content(n: torch.Tensor, is_gc: torch.Tensor) -> torch.Tensor:
    """GC_w = n_{wa} g_a / Σ_a n_{wa}  (a summed), with g = (0, 1, 1, 0).   (W, 4) → (W,)"""
    return torch.einsum("wa,a->w", n, is_gc) / n.sum(dim=1)


def gc_skew(n: torch.Tensor, is_gc: torch.Tensor, g_minus_c: torch.Tensor) -> torch.Tensor:
    """skew_w = (G - C) / (G + C) = n_{wa} c_a / n_{wa} g_a  (a summed).   (W, 4) → (W,)"""
    return torch.einsum("wa,a->w", n, g_minus_c) / torch.einsum("wa,a->w", n, is_gc).clamp(min=1)


def cumulative_skew(X: torch.Tensor, g_minus_c: torch.Tensor) -> torch.Tensor:
    """S_n = Σ_{n' ≤ n} X_{n'a} c_a: running count of G minus C along the chromosome.   (L, 4) → (L,)"""
    d = torch.einsum("na,a->n", X, g_minus_c)  # +1 for G, -1 for C, 0 for A and T
    return torch.cumsum(d, dim=0)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/kmer_gc_skew/tensors.py
"""Build the concrete tensors: a synthetic circular genome, its one-hot encoding and the k-mer place values.

Bases are numbered A = 0, C = 1, G = 2, T = 3, so the complement of base a is 3 - a.
"""

import torch

BASES = "ACGT"
ISLANDS = [(0.62, -0.12), (0.85, -0.10)]  # (start as a fraction of the genome, change of GC content)


def synthetic_genome(
    length: int, origin_frac: float, gc: float, skew: float, seed: int, device: torch.device
) -> tuple[torch.Tensor, int, int]:
    """A random circular chromosome with a planted replication origin and terminus.

    From the origin clockwise to the terminus (half the circle) is the leading strand, where G is more common than C
    by the factor (1 + skew) / (1 - skew); on the other half C and G swap. Two short "islands" have a lower GC content.
    Returns seq (L,) with base indices 0..3, the origin position and the terminus position.
    """
    origin = int(origin_frac * length) % length
    terminus = (origin + length // 2) % length
    pos = torch.arange(length)
    leading = (pos - origin) % length < length // 2  # (L,) True on the half from origin to terminus

    gc_n = torch.full((length,), gc)  # (L,) GC content at every position
    for start, delta in ISLANDS:
        a = int(start * length)
        gc_n[a : a + length // 25] += delta
    s = torch.where(leading, skew, -skew)  # (L,) +skew on the leading half, -skew on the lagging half

    # probabilities of A, C, G, T at each position: (L, 4)
    prob = torch.stack([(1 - gc_n) / 2, gc_n / 2 * (1 - s), gc_n / 2 * (1 + s), (1 - gc_n) / 2], dim=1)
    u = torch.rand(length, 1, generator=torch.Generator().manual_seed(seed))
    seq = (u > prob.cumsum(dim=1)).sum(dim=1).clamp(max=3)  # inverse-CDF sampling: one base per position
    return seq.to(device), origin, terminus


def one_hot(seq: torch.Tensor) -> torch.Tensor:
    """X_{na} = 1 if base n is letter a, else 0.   (L,) → (L, 4)."""
    return torch.nn.functional.one_hot(seq, num_classes=4).float()


def place_values(k: int, device: torch.device) -> torch.Tensor:
    """P_{ja} = a · 4^(k-1-j): reading a k-mer as a number in base 4 (AAA = 0, AAC = 1, …, TTT = 4^k - 1).   (k, 4)"""
    digit = torch.arange(4, device=device, dtype=torch.float32)  # value of letter a
    weight = 4.0 ** torch.arange(k - 1, -1, -1, device=device, dtype=torch.float32)  # 4^(k-1-j)
    return torch.einsum("j,a->ja", weight, digit)


def kmer_names(k: int) -> list[str]:
    """All 4^k words in the same order as the k-mer index."""
    names = [""]
    for _ in range(k):
        names = [w + b for w in names for b in BASES]
    return names


def gc_strand_weights(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """g_a = [a ∈ {G, C}] and c_a = [a = G] - [a = C]: the per-base GC indicator and the skew sign.   (4,), (4,)"""
    is_gc = torch.tensor([0.0, 1.0, 1.0, 0.0], device=device)
    g_minus_c = torch.tensor([0.0, -1.0, 1.0, 0.0], device=device)
    return is_gc, g_minus_c


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/kmer_gc_skew/solve.py
"""Analyse the genome: k-mer spectrum, GC content and skew in sliding windows, origin from the cumulative skew."""

import torch

from . import equation


def count_kmers(X: torch.Tensor, P: torch.Tensor) -> torch.Tensor:
    """k-mer spectrum (4^k,) of a one-hot sequence X (L, 4); the counts add up to L - k + 1."""
    k = P.shape[0]
    U = equation.unfold(X, k)  # (L - k + 1, 4, k)
    return equation.kmer_counts(equation.kmer_index(U, P), k)


def windows(X: torch.Tensor, size: int, is_gc: torch.Tensor, g_minus_c: torch.Tensor) -> dict[str, torch.Tensor]:
    """GC content and skew of windows of `size` bases that start every size/2 bases around the circle."""
    step = max(1, size // 2)
    n = equation.window_base_counts(equation.circular(X, size, step), size, step)  # (W, 4)
    skew = equation.gc_skew(n, is_gc, g_minus_c)  # (W,)
    return {
        "start": torch.arange(n.shape[0], device=X.device) * step,  # (W,) first base of each window
        "gc": equation.gc_content(n, is_gc),  # (W,)
        "skew": skew,
        "cum_skew": torch.cumsum(skew, dim=0),  # (W,) the classic windowed cumulative skew plot
    }


def find_origin(X: torch.Tensor, g_minus_c: torch.Tensor) -> tuple[torch.Tensor, int, int]:
    """Per-base cumulative skew S (L,), and the predicted origin (argmin S) and terminus (argmax S)."""
    S = equation.cumulative_skew(X, g_minus_c)
    return S, int(S.argmin()), int(S.argmax())


def circular_distance(a: int, b: int, length: int) -> int:
    """Distance between two positions on a circle of `length` bases."""
    d = abs(a - b) % length
    return min(d, length - d)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/kmer_gc_skew/plot.py
"""Animate a window walking round the circular chromosome: skew ring, cumulative skew curve and the k-mer spectrum."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgba_array

from problems._common.plotting import frame_indices, save_gif

POS, NEG, GC, ORI, TER = "#2563eb", "#ea580c", "#16a34a", "#dc2626", "#9333ea"


def animate(
    win: dict[str, torch.Tensor],
    S: torch.Tensor,
    counts: torch.Tensor,
    names: list[str],
    L: int,
    planted: tuple[int, int],
    found: tuple[int, int],
    path: Path,
) -> Path:
    start = win["start"].cpu().numpy()
    skew, gc = win["skew"].cpu().numpy(), win["gc"].cpu().numpy()
    n_win = len(start)
    step = max(1, L // 2000)  # ~2000 points are plenty for drawing the per-base curve
    pos_kb = np.arange(0, L, step) / 1e3
    S_np = S[::step].cpu().numpy()

    fig = plt.figure(figsize=(8.4, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.15], height_ratios=[1.2, 1])
    ax_c = fig.add_subplot(grid[:, 0])
    ax_s = fig.add_subplot(grid[0, 1])
    ax_k = fig.add_subplot(grid[1, 1])

    # --- the circular chromosome: skew bars on the outer ring, GC deviation on the inner ring ---
    ax_c.set_aspect("equal")
    ax_c.set_xlim(-1.62, 1.62)
    ax_c.set_ylim(-1.62, 1.62)
    ax_c.axis("off")
    a0 = 2 * np.pi * start / L  # angle where each window starts (clockwise from the top)
    a1 = 2 * np.pi * np.minimum(start + start[1], L) / L
    s_scale = 0.35 / max(np.abs(skew).max(), 1e-9)
    g_scale = 0.25 / max(np.abs(gc - gc.mean()).max(), 1e-9)
    skew_ring = PolyCollection(_wedges(a0, a1, 1.05, 1.05 + skew * s_scale), lw=0)
    gc_ring = PolyCollection(_wedges(a0, a1, 0.62, 0.62 + (gc - gc.mean()) * g_scale), lw=0)
    skew_rgba = to_rgba_array(np.where(skew > 0, POS, NEG))  # (W, 4)
    gc_rgba = to_rgba_array([GC] * n_win)
    ax_c.add_collection(skew_ring)
    ax_c.add_collection(gc_ring)
    ring = np.linspace(0, 2 * np.pi, 200)
    ax_c.plot(1.05 * np.sin(ring), 1.05 * np.cos(ring), color="#374151", lw=1)
    ax_c.plot(0.62 * np.sin(ring), 0.62 * np.cos(ring), color="#9ca3af", lw=0.8)
    for q in range(4):  # position ticks every quarter of the genome
        a = q * np.pi / 2
        ax_c.text(1.58 * np.sin(a), 1.58 * np.cos(a), f"{q * L / 4e3:.0f} kb", ha="center", va="center", fontsize=8)
    for p, color, label in [(planted[0], ORI, "ori"), (planted[1], TER, "ter")]:
        a = 2 * np.pi * p / L
        ax_c.plot([1.45 * np.sin(a)], [1.45 * np.cos(a)], marker=(3, 0, 180 - np.degrees(a)), color=color, ms=7)
        ax_c.text(1.3 * np.sin(a), 1.3 * np.cos(a), label, color=color, ha="center", va="center", fontsize=8)
    ax_c.text(0, 0, "GC skew (outer)\nGC content (inner)", ha="center", va="center", fontsize=7.5, color="#6b7280")
    pointer = ax_c.plot([], [], color="#374151", lw=1.2)[0]
    ax_c.set_title("circular chromosome", fontsize=10)

    # --- cumulative skew S_n, drawn up to the window ---
    ax_s.set_xlim(0, L / 1e3)
    pad = 0.08 * (S_np.max() - S_np.min() + 1)
    ax_s.set_ylim(S_np.min() - pad, S_np.max() + pad)
    ax_s.set_xlabel("position (kb)", labelpad=1)
    ax_s.set_ylabel("cumulative G − C")
    ax_s.set_title("cumulative GC skew: min = origin, max = terminus", fontsize=10)
    for p, color in [(planted[0], ORI), (planted[1], TER)]:
        ax_s.axvline(p / 1e3, color=color, ls="--", lw=1, alpha=0.7)
    curve = ax_s.plot([], [], color="#374151", lw=1.4)[0]
    found_marks = ax_s.plot(
        [found[0] / 1e3, found[1] / 1e3], [S_np.min(), S_np.max()], "o", ms=7, mfc="none", mew=1.8, color="#111827"
    )[0]
    found_label = ax_s.text(0.98, 0.06, "", transform=ax_s.transAxes, fontsize=8, ha="right")

    # --- k-mer spectrum: one vertical line per word, coloured by its number of G/C letters ---
    c = counts.cpu().numpy()
    k = len(names[0])
    n_gc = np.array([sum(ch in "GC" for ch in w) for w in names])
    cmap = plt.get_cmap("viridis", k + 1)
    ax_k.vlines(np.arange(len(c)), 0, c, colors=cmap(n_gc), lw=max(0.5, 220 / len(c)))
    ax_k.set_xlim(-1, len(c))
    ax_k.set_ylim(0, c.max() * 1.15)
    ax_k.set_ylabel("count", labelpad=1)
    ax_k.set_xlabel(f"{k}-mer index ({names[0]} … {names[-1]})", labelpad=1)
    ax_k.set_title(f"{k}-mer spectrum, colour = number of G/C (0–{k})", fontsize=10)
    ax_k.grid(axis="x", visible=False)

    def update(i: int):
        upto = start[i] + start[1]
        shown = (np.arange(n_win) <= i)[:, None]  # windows the pointer has passed
        skew_ring.set_facecolor(np.where(shown, skew_rgba, 0.0))  # alpha 0 hides the rest
        gc_ring.set_facecolor(np.where(shown, gc_rgba, 0.0))
        a = 2 * np.pi * upto / L
        pointer.set_data([0.4 * np.sin(a), 1.45 * np.sin(a)], [0.4 * np.cos(a), 1.45 * np.cos(a)])
        m = pos_kb <= upto / 1e3
        curve.set_data(pos_kb[m], S_np[m])
        done = i == n_win - 1
        found_marks.set_visible(done)
        found_label.set_text(f"○ found: ori {found[0] / 1e3:.1f} kb, ter {found[1] / 1e3:.1f} kb" if done else "")

    fig.tight_layout()
    frames = frame_indices(n_win, 60)
    frames += [frames[-1]] * (len(frames) // 6)  # hold the final picture for a moment
    return save_gif(fig, update, frames, path, fps=15)


def _wedges(a0: np.ndarray, a1: np.ndarray, r0: float, r1: np.ndarray) -> list[np.ndarray]:
    """One quadrilateral per window between radii r0 and r1[w], angles measured clockwise from the top."""
    out = []
    for w in range(len(a0)):
        a = np.array([a0[w], a1[w], a1[w], a0[w]])
        r = np.array([r0, r0, r1[w], r1[w]])
        out.append(np.stack([r * np.sin(a), r * np.cos(a)], axis=1))
    return out


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'length_kb': 200,
    'k': 3,
    'window': 2000,
    'origin': 0.3,
    'skew': 0.05,
    'gc': 0.5,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.kmer_gc_skew")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.kmer_gc_skew.main")
problem_dir = Path("problems/bioinformatics/kmer_gc_skew")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
